# MilletSaarthi — Grain Classifier (Agent 1)
MobileNetV3-Large, 7 classes, transfer learning.
Leak-free split by original grain (using `aug_XXXX_N.jpg` grouping).

## 1. Setup & mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Copy dataset to local SSD (much faster than Drive)

In [ ]:
import shutil, os, time
SRC = '/content/drive/MyDrive/MilletSaarthi/data/dataset_augmented'
DST = '/content/data/dataset_augmented'
if not os.path.exists(DST):
    t=time.time(); shutil.copytree(SRC, DST); print(f'Copied in {time.time()-t:.1f}s')
else:
    print('Already copied')
print(os.listdir(DST))

## 2b. Add "other" class — non-grain images (cars, faces, animals, etc.)
Download ~300 diverse images from CIFAR-100 so the model learns to reject non-millet uploads.

In [ ]:
import torchvision, os, random
from PIL import Image as PILImage

OTHER_DIR = os.path.join(DST, 'other')
NUM_OTHER = 300  # number of non-grain images to collect

if not os.path.exists(OTHER_DIR):
    os.makedirs(OTHER_DIR)

    # Download CIFAR-100 (100 categories: cars, people, animals, buildings, etc.)
    cifar = torchvision.datasets.CIFAR100(
        root='/content/cifar100', train=True, download=True
    )

    # Pick NUM_OTHER random indices spread across many categories
    random.seed(42)
    indices = random.sample(range(len(cifar)), NUM_OTHER)

    for i, idx in enumerate(indices):
        img, _ = cifar[idx]  # img is a 32x32 PIL image
        # Upscale to 224x224 to match grain image resolution
        img = img.resize((224, 224), PILImage.BICUBIC)
        img.save(os.path.join(OTHER_DIR, f'other_{i:04d}.jpg'))

    print(f'Created {NUM_OTHER} non-grain images in {OTHER_DIR}')
else:
    print(f'other/ already exists with {len(os.listdir(OTHER_DIR))} images')

print('Updated classes:', sorted(os.listdir(DST)))

## 3. Leak-free split by source grain

In [ ]:
import os, re, random, csv
from collections import defaultdict
random.seed(42)

DATA_DIR = '/content/data/dataset_augmented'
CLASSES = sorted(os.listdir(DATA_DIR))
print('Classes:', CLASSES)
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}

train_rows, val_rows, test_rows = [], [], []
pat = re.compile(r'aug_(\d+)_\d+')

for cname in CLASSES:
    cdir = os.path.join(DATA_DIR, cname)
    groups = defaultdict(list)
    for f in os.listdir(cdir):
        m = pat.match(f)
        key = m.group(1) if m else f  # fall back to filename if pattern fails
        groups[key].append(os.path.join(cdir, f))
    keys = list(groups.keys()); random.shuffle(keys)
    n = len(keys); n_tr = int(0.70*n); n_val = int(0.15*n)
    tr_k, val_k, te_k = keys[:n_tr], keys[n_tr:n_tr+n_val], keys[n_tr+n_val:]
    idx = CLASS_TO_IDX[cname]
    for k in tr_k:  [train_rows.append((p, idx)) for p in groups[k]]
    for k in val_k: [val_rows.append((p, idx))   for p in groups[k]]
    for k in te_k:  [test_rows.append((p, idx))  for p in groups[k]]
    print(f'{cname}: {n} groups -> tr {len(tr_k)} val {len(val_k)} test {len(te_k)}')

print(f'\nTotal: train {len(train_rows)}  val {len(val_rows)}  test {len(test_rows)}')

os.makedirs('/content/splits', exist_ok=True)
for name, rows in [('train',train_rows),('val',val_rows),('test',test_rows)]:
    with open(f'/content/splits/{name}.csv','w',newline='') as f:
        w=csv.writer(f); w.writerow(['filepath','label']); w.writerows(rows)

## 4. Dataset & dataloaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import csv

IMG = 224
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG,IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class GrainDS(Dataset):
    def __init__(self, csv_path, tf):
        self.rows=[]
        with open(csv_path) as f:
            r=csv.reader(f); next(r)
            for p,l in r: self.rows.append((p,int(l)))
        self.tf=tf
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        p,l=self.rows[i]
        return self.tf(Image.open(p).convert('RGB')), l

BS=64
train_ds=GrainDS('/content/splits/train.csv',train_tf)
val_ds  =GrainDS('/content/splits/val.csv',  eval_tf)
test_ds =GrainDS('/content/splits/test.csv', eval_tf)
train_dl=DataLoader(train_ds,BS,shuffle=True, num_workers=2,pin_memory=True)
val_dl  =DataLoader(val_ds,  BS,shuffle=False,num_workers=2,pin_memory=True)
test_dl =DataLoader(test_ds, BS,shuffle=False,num_workers=2,pin_memory=True)
print(len(train_ds),len(val_ds),len(test_ds))

## 5. Model — MobileNetV3-Large

In [ ]:
import torch.nn as nn
from torchvision import models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = len(CLASSES)

model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
in_feat = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_feat, NUM_CLASSES)
model = model.to(device)

def set_backbone_trainable(flag):
    for p in model.features.parameters(): p.requires_grad = flag


## 6. Train — Phase 1 (frozen backbone) → Phase 2 (fine-tune)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

def run_epoch(dl, train=True):
    model.train() if train else model.eval()
    tot_loss=tot=correct=0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x,y in dl:
            x,y=x.to(device),y.to(device)
            if train: optimizer.zero_grad()
            out=model(x); loss=criterion(out,y)
            if train: loss.backward(); optimizer.step()
            tot_loss += loss.item()*x.size(0)
            correct  += (out.argmax(1)==y).sum().item()
            tot      += x.size(0)
    return tot_loss/tot, correct/tot

best_val=0; best_path='/content/best_model.pth'

# Phase 1: frozen backbone
set_backbone_trainable(False)
optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
print('--- Phase 1: head only ---')
for e in range(5):
    tl,ta = run_epoch(train_dl, True)
    vl,va = run_epoch(val_dl,  False)
    print(f'E{e+1}  train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}')
    if va>best_val: best_val=va; torch.save(model.state_dict(),best_path)

# Phase 2: fine-tune all
set_backbone_trainable(True)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)
print('--- Phase 2: fine-tune ---')
patience=4; bad=0
for e in range(15):
    tl,ta = run_epoch(train_dl, True)
    vl,va = run_epoch(val_dl,  False)
    scheduler.step()
    print(f'E{e+1}  train {tl:.3f}/{ta:.3f}  val {vl:.3f}/{va:.3f}')
    if va>best_val: best_val=va; torch.save(model.state_dict(),best_path); bad=0
    else: bad+=1
    if bad>=patience: print('Early stop'); break

print(f'Best val acc: {best_val:.4f}')

## 7. Evaluate on test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()
ys,ps=[],[]
with torch.no_grad():
    for x,y in test_dl:
        p = model(x.to(device)).argmax(1).cpu().numpy()
        ys += y.tolist(); ps += p.tolist()

print(classification_report(ys,ps,target_names=CLASSES,digits=4))
print('Confusion matrix:'); print(confusion_matrix(ys,ps))

## 8. Export — ONNX + save to Drive

In [ ]:
import json, shutil
OUT='/content/drive/MyDrive/MilletSaarthi/models'
os.makedirs(OUT, exist_ok=True)

shutil.copy('/content/best_model.pth', f'{OUT}/mobilenetv3_best.pth')

dummy = torch.randn(1,3,IMG,IMG,device=device)
torch.onnx.export(model, dummy, f'{OUT}/mobilenetv3_best.onnx',
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input':{0:'batch'},'logits':{0:'batch'}},
                  opset_version=17)

with open(f'{OUT}/classes.json','w') as f: json.dump(CLASSES,f,indent=2)
print('Saved to', OUT)

## 9. Quick inference test

In [ ]:
def predict(path):
    img = eval_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(img),1)[0].cpu().numpy()
    i = probs.argmax()
    return CLASSES[i], float(probs[i])

sample = test_ds.rows[0][0]
print(sample, '->', predict(sample))